# LeetCode 3383 — Minimum Runes to Add to Cast Spell bằng Python

- Link: https://leetcode.com/problems/minimum-runes-to-add-to-cast-spell/
- Độ khó: **Hard** · **Premium**
- Chủ đề: **Đồ thị có hướng**, **thành phần liên thông mạnh (SCC)** — Tarjan/Kosaraju không đệ quy, **đồ thị cô đặc (condensation)**, đếm **SCC nguồn** không chứa tinh thể

> **Ghi chú Premium:** bài này là Premium nên LeetCode không trả về đề qua API công khai. Đề, ví dụ, ràng buộc và chữ ký `minRunesToAdd(self, n, crystals, flowFrom, flowTo)` lấy theo trang doocs (https://leetcode.doocs.org/en/lc/3383/). Vui lòng **đối chiếu lại** nếu có tài khoản Premium.

Skeleton:

```python
class Solution:
    def minRunesToAdd(self, n: int, crystals: List[int], flowFrom: List[int], flowTo: List[int]) -> int:
```

## 1. Mục tiêu học

1. Hiểu yêu cầu: sau khi thêm cạnh, **mọi đỉnh đều nhận được năng lượng** — tức là **đi tới được từ một đỉnh có tinh thể**.
2. Gom đồ thị thành các **SCC**; trong đồ thị cô đặc (DAG), một SCC được "thắp sáng" ⇔ nó chứa tinh thể hoặc có SCC thắp sáng trỏ tới.
3. Chứng minh đáp án = **số SCC có bậc vào 0 (trong đồ thị cô đặc) và không chứa tinh thể**.
4. Cài **Tarjan không đệ quy** (và Kosaraju) để chịu được `n = 10^5`.
5. So sánh với lời giải doocs: lan truyền từ tinh thể + duyệt theo thứ tự kết thúc DFS ngược.

## 2. Đề bài & ví dụ

Có `n` điểm, một số điểm chứa **tinh thể** (`crystals`). Các cạnh có hướng `flowFrom[i] → flowTo[i]` truyền năng lượng. Thêm ít cạnh có hướng nhất sao cho **mỗi điểm** hoặc chứa tinh thể, hoặc **nhận** năng lượng (đi tới được từ một tinh thể theo các cạnh).

**Ví dụ 1:** `n = 6, crystals = [0], cạnh 0→1→2→3→0` → `2` (thêm `0→4`, `0→5`).

**Ví dụ 2:** `n = 7, crystals = [3,5], cạnh 0→1, 1→2, 2→0, 3→4, 5→6` → `1` (thêm `4→2`).

**Ràng buộc (theo doocs):** `n ≤ 10^5`, số cạnh ≤ `2·10^5`, không có cạnh lặp/khuyên.

In [1]:
from typing import List
from collections import deque
import itertools

examples = [
    (6, [0], [0, 1, 2, 3], [1, 2, 3, 0], 2),
    (7, [3, 5], [0, 1, 2, 3, 5], [1, 2, 0, 4, 6], 1),
]
for n, cr, ff, ft, exp in examples:
    print('n =', n, 'tinh thể', cr, 'cạnh', list(zip(ff, ft)), '-> kỳ vọng', exp)

n = 6 tinh thể [0] cạnh [(0, 1), (1, 2), (2, 3), (3, 0)] -> kỳ vọng 2
n = 7 tinh thể [3, 5] cạnh [(0, 1), (1, 2), (2, 0), (3, 4), (5, 6)] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Tập được thắp sáng: BFS đa nguồn

In [2]:
def lit_set(n, crystals, edges):
    g = [[] for _ in range(n)]
    for a, b in edges:
        g[a].append(b)
    vis = [False] * n
    q = deque(crystals)
    for x in crystals:
        vis[x] = True
    while q:
        a = q.popleft()
        for b in g[a]:
            if not vis[b]:
                vis[b] = True; q.append(b)
    return vis

n, cr, ff, ft, _ = examples[1]
print('được thắp sáng:', [i for i, v in enumerate(lit_set(n, cr, list(zip(ff, ft)))) if v])

được thắp sáng: [3, 4, 5, 6]


### 3.2. SCC và đồ thị cô đặc

Hai đỉnh cùng SCC nếu đi được lẫn nhau. Gom mỗi SCC thành một đỉnh ⇒ đồ thị **không chu trình** (DAG). Trong DAG, mọi đỉnh đi tới được từ ít nhất một **nguồn** (bậc vào 0).

### 3.3. Tarjan không đệ quy

DFS giữ `index`, `low`, ngăn xếp; khi `low[v] == index[v]` thì bật ra một SCC. Thay đệ quy bằng ngăn xếp `(đỉnh, vị trí cạnh kế tiếp)`.

In [3]:
def tarjan_scc(n, g):
    index = [-1] * n; low = [0] * n; onst = [False] * n
    st, comp = [], [-1] * n
    idx = c = 0
    for s in range(n):
        if index[s] != -1:
            continue
        index[s] = low[s] = idx; idx += 1; st.append(s); onst[s] = True
        work = [(s, 0)]
        while work:
            v, pi = work[-1]
            if pi < len(g[v]):
                work[-1] = (v, pi + 1)
                w = g[v][pi]
                if index[w] == -1:
                    index[w] = low[w] = idx; idx += 1; st.append(w); onst[w] = True
                    work.append((w, 0))
                elif onst[w]:
                    low[v] = min(low[v], index[w])
            else:
                work.pop()
                if work:
                    u = work[-1][0]; low[u] = min(low[u], low[v])
                if low[v] == index[v]:
                    while True:
                        w = st.pop(); onst[w] = False; comp[w] = c
                        if w == v:
                            break
                    c += 1
    return c, comp

g = [[] for _ in range(7)]
for a, b in zip(ff, ft):
    g[a].append(b)
print(tarjan_scc(7, g))

(5, [0, 0, 0, 2, 1, 4, 3])


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2: các SCC là `{0, 1, 2}` (chu trình), `{3}`, `{4}`, `{5}`, `{6}`. Đồ thị cô đặc: `{3} → {4}`, `{5} → {6}`; `{0,1,2}` không có cạnh vào.

- Nguồn: `{0,1,2}`, `{3}`, `{5}`. Nguồn `{3}`, `{5}` chứa tinh thể. Nguồn `{0,1,2}` không ⇒ **phải** thêm ít nhất một cạnh trỏ vào nó; một cạnh (từ đỉnh đã sáng, ví dụ `4 → 2`) là đủ vì trong SCC mọi đỉnh đi tới nhau.

⇒ đáp án 1.

## 5. Cách 1 — Brute Force: thử thêm mọi tập cạnh nhỏ

In [4]:
def min_runes_brute(n, crystals, ff, ft):
    E = list(zip(ff, ft)); have = set(E)
    cand = [(a, b) for a in range(n) for b in range(n) if a != b and (a, b) not in have]
    for k in range(n + 1):
        for add in itertools.combinations(cand, k):
            if all(lit_set(n, crystals, E + list(add))):
                return k

for n, cr, ff, ft, exp in examples:
    print(min_runes_brute(n, cr, ff, ft), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1


**Độ phức tạp:** mũ theo số cạnh thêm — chỉ cho `n ≤ 6`.

## 6. Ý tưởng thuật toán / bước đột phá

**Định lý.** Đáp án = số SCC `K` có bậc vào 0 trong đồ thị cô đặc và **không chứa tinh thể**.

- *Cận dưới:* SCC nguồn không có tinh thể không nhận năng lượng từ đâu cả ⇒ phải có ít nhất một cạnh **mới** trỏ vào nó; các SCC nguồn khác nhau cần các cạnh khác nhau (mỗi cạnh mới chỉ có một đích).
- *Đạt được:* thêm từ một đỉnh tinh thể bất kỳ tới một đỉnh của mỗi SCC nguồn như vậy. Mọi SCC khác hoặc là nguồn có tinh thể, hoặc đi tới được từ một nguồn (DAG) ⇒ mọi đỉnh đều sáng.

Dùng Tarjan `O(n + m)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Lời giải doocs: lan truyền + thứ tự kết thúc DFS ngược

BFS đánh dấu các đỉnh sáng từ tinh thể. Trên phần chưa sáng, DFS ghi thứ tự kết thúc; duyệt theo thứ tự **ngược** (đỉnh của SCC nguồn đi trước). Gặp đỉnh còn tối ⇒ thêm một cạnh (`ans += 1`) và BFS lan từ đỉnh đó.

In [5]:
def min_runes_doocs(n, crystals, ff, ft):
    g = [[] for _ in range(n)]
    for a, b in zip(ff, ft):
        g[a].append(b)
    vis = [0] * n                     # 0 = tối, 1 = sáng, 2 = đã duyệt DFS (còn tối)
    def bfs(start):
        q = deque(start)
        while q:
            a = q.popleft()
            for b in g[a]:
                if vis[b] != 1:
                    vis[b] = 1; q.append(b)
    for x in crystals:
        vis[x] = 1
    bfs(crystals)
    order = []
    for s in range(n):
        if vis[s] == 0:
            vis[s] = 2
            stack = [(s, 0)]
            while stack:
                v, pi = stack[-1]
                if pi < len(g[v]):
                    stack[-1] = (v, pi + 1)
                    w = g[v][pi]
                    if vis[w] == 0:
                        vis[w] = 2; stack.append((w, 0))
                else:
                    stack.pop(); order.append(v)
    ans = 0
    for v in reversed(order):
        if vis[v] == 2:
            vis[v] = 1; bfs([v]); ans += 1
    return ans

for n, cr, ff, ft, exp in examples:
    print(min_runes_doocs(n, cr, ff, ft), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1


### 7.2. Cách 3 — Tarjan + đếm SCC nguồn không có tinh thể (lời giải chính)

In [6]:
def min_runes_tarjan(n, crystals, ff, ft):
    g = [[] for _ in range(n)]
    for a, b in zip(ff, ft):
        g[a].append(b)
    c, comp = tarjan_scc(n, g)
    indeg = [0] * c
    has_crystal = [False] * c
    for a, b in zip(ff, ft):
        if comp[a] != comp[b]:
            indeg[comp[b]] += 1
    for x in crystals:
        has_crystal[comp[x]] = True
    return sum(1 for k in range(c) if indeg[k] == 0 and not has_crystal[k])

for n, cr, ff, ft, exp in examples:
    print(min_runes_tarjan(n, cr, ff, ft), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1


### 7.3. Cách 4 — Kosaraju không đệ quy

DFS lần 1 lấy thứ tự kết thúc; DFS lần 2 trên đồ thị **ngược** theo thứ tự kết thúc giảm dần, mỗi lần gom một SCC.

In [7]:
def kosaraju_scc(n, ff, ft):
    g = [[] for _ in range(n)]; rg = [[] for _ in range(n)]
    for a, b in zip(ff, ft):
        g[a].append(b); rg[b].append(a)
    seen = [False] * n; order = []
    for s in range(n):
        if seen[s]:
            continue
        seen[s] = True; stack = [(s, 0)]
        while stack:
            v, pi = stack[-1]
            if pi < len(g[v]):
                stack[-1] = (v, pi + 1); w = g[v][pi]
                if not seen[w]:
                    seen[w] = True; stack.append((w, 0))
            else:
                stack.pop(); order.append(v)
    comp = [-1] * n; c = 0
    for s in reversed(order):
        if comp[s] != -1:
            continue
        comp[s] = c; stack = [s]
        while stack:
            v = stack.pop()
            for w in rg[v]:
                if comp[w] == -1:
                    comp[w] = c; stack.append(w)
        c += 1
    return c, comp

def min_runes_kosaraju(n, crystals, ff, ft):
    c, comp = kosaraju_scc(n, ff, ft)
    indeg = [0] * c; has = [False] * c
    for a, b in zip(ff, ft):
        if comp[a] != comp[b]:
            indeg[comp[b]] += 1
    for x in crystals:
        has[comp[x]] = True
    return sum(1 for k in range(c) if indeg[k] == 0 and not has[k])

for n, cr, ff, ft, exp in examples:
    print(min_runes_kosaraju(n, cr, ff, ft), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 2: in SCC, bậc vào trong đồ thị cô đặc và cờ tinh thể.

In [8]:
def trace(n, crystals, ff, ft):
    g = [[] for _ in range(n)]
    for a, b in zip(ff, ft):
        g[a].append(b)
    c, comp = tarjan_scc(n, g)
    members = [[v for v in range(n) if comp[v] == k] for k in range(c)]
    indeg = [0] * c
    for a, b in zip(ff, ft):
        if comp[a] != comp[b]:
            indeg[comp[b]] += 1
    ans = 0
    for k in range(c):
        hc = any(x in members[k] for x in crystals)
        need = indeg[k] == 0 and not hc
        ans += need
        print(f"SCC {members[k]}: bậc vào {indeg[k]}, có tinh thể: {hc} -> {'CẦN 1 cạnh' if need else '-'}")
    print('đáp án:', ans)

trace(*examples[1][:4])

SCC [0, 1, 2]: bậc vào 0, có tinh thể: False -> CẦN 1 cạnh
SCC [4]: bậc vào 1, có tinh thể: False -> -
SCC [3]: bậc vào 0, có tinh thể: True -> -
SCC [6]: bậc vào 1, có tinh thể: False -> -
SCC [5]: bậc vào 0, có tinh thể: True -> -
đáp án: 1


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def minRunesToAdd(self, n: int, crystals: List[int], flowFrom: List[int], flowTo: List[int]) -> int:
        g = [[] for _ in range(n)]
        for a, b in zip(flowFrom, flowTo):
            g[a].append(b)
        index = [-1] * n
        low = [0] * n
        onst = [False] * n
        comp = [-1] * n
        st = []
        idx = c = 0
        for s in range(n):
            if index[s] != -1:
                continue
            index[s] = low[s] = idx
            idx += 1
            st.append(s)
            onst[s] = True
            work = [[s, 0]]
            while work:
                top = work[-1]
                v = top[0]
                if top[1] < len(g[v]):
                    w = g[v][top[1]]
                    top[1] += 1
                    if index[w] == -1:
                        index[w] = low[w] = idx
                        idx += 1
                        st.append(w)
                        onst[w] = True
                        work.append([w, 0])
                    elif onst[w] and index[w] < low[v]:
                        low[v] = index[w]
                else:
                    work.pop()
                    if work:
                        u = work[-1][0]
                        if low[v] < low[u]:
                            low[u] = low[v]
                    if low[v] == index[v]:
                        while True:
                            w = st.pop()
                            onst[w] = False
                            comp[w] = c
                            if w == v:
                                break
                        c += 1
        has_in = [False] * c
        for a, b in zip(flowFrom, flowTo):
            if comp[a] != comp[b]:
                has_in[comp[b]] = True
        for x in crystals:
            has_in[comp[x]] = True
        return has_in.count(False)

for n, cr, ff, ft, exp in examples:
    got = Solution().minRunesToAdd(n, cr, ff, ft)
    print(got, 'OK' if got == exp else 'SAI')

2 OK
1 OK


Bản nộp gộp hai điều kiện "có cạnh vào từ SCC khác" và "có tinh thể" vào một mảng `has_in`: SCC cần thêm cạnh ⇔ cả hai đều sai.

## 10. Vì sao thuật toán đúng

1. **Thắp sáng ⇔ đi tới được từ tinh thể.** Trong đồ thị cô đặc (DAG), một SCC được thắp sáng ⇔ nó chứa tinh thể hoặc có đường từ SCC chứa tinh thể tới nó.
2. **Cận dưới:** mỗi SCC nguồn `S` không có tinh thể: không có cạnh cũ nào đi vào `S`, nên phải có một cạnh **mới** với đích trong `S`. Các SCC nguồn khác nhau rời nhau ⇒ cần ít nhất `K` cạnh mới.
3. **Đạt được:** thêm cạnh từ một tinh thể tới một đỉnh của mỗi nguồn như vậy. Mỗi SCC còn lại hoặc là nguồn có tinh thể, hoặc có tổ tiên là một nguồn (vì DAG hữu hạn) — nguồn đó giờ đã sáng ⇒ mọi SCC sáng, và trong SCC mọi đỉnh đi tới nhau.
4. **Cách 2:** thứ tự kết thúc DFS ngược bảo đảm gặp một đỉnh của mỗi SCC nguồn (trong phần tối) trước mọi đỉnh mà nguồn đó đi tới; mỗi lần thêm cạnh thắp sáng đúng một nguồn mới cùng phần nó đi tới ⇒ đếm đúng `K`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử thêm tập cạnh | mũ | `O(n²)` |
| 2. Lan truyền + thứ tự DFS | `O(n + m)` | `O(n + m)` |
| 3. Tarjan | `O(n + m)` | `O(n + m)` |
| 4. Kosaraju | `O(n + m)` | `O(n + m)` (thêm đồ thị ngược) |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'doocs': min_runes_doocs, 'tarjan': min_runes_tarjan, 'kosaraju': min_runes_kosaraju,
         'submit': lambda n, c, a, b: Solution().minRunesToAdd(n, c, a, b)}
edge = [
    (2, [0], [0], [1], 0),
    (2, [1], [0], [1], 1),             # 0 không nhận được gì
    (3, [0, 1, 2], [0], [1], 0),       # mọi điểm có tinh thể
    (4, [0], [1, 2], [2, 1], 2),       # SCC {1,2} và đỉnh 3 cô lập
    (4, [3], [0, 1, 2], [1, 2, 0], 1),
]
for n, cr, ff, ft, exp in examples + edge:
    assert min_runes_brute(n, cr, ff, ft) == exp, (n, cr, ff, ft, min_runes_brute(n, cr, ff, ft))
    for name, f in impls.items():
        assert f(n, cr, ff, ft) == exp, (name, n)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3383)
cnt = 0
for _ in range(300):
    n = rng.randint(2, 5)
    pairs = [(a, b) for a in range(n) for b in range(n) if a != b]
    E = rng.sample(pairs, rng.randint(1, min(len(pairs), 6)))
    cr = rng.sample(range(n), rng.randint(1, n))
    ff, ft = [a for a, _ in E], [b for _, b in E]
    exp = min_runes_brute(n, cr, ff, ft)
    for name, f in impls.items():
        assert f(n, cr, ff, ft) == exp, (name, n, cr, E)
    cnt += 1
for _ in range(200):
    n = rng.randint(2, 60)
    m = rng.randint(1, 3 * n)
    E = list({(rng.randrange(n), rng.randrange(n)) for _ in range(m)})
    E = [(a, b) for a, b in E if a != b] or [(0, 1)]
    cr = rng.sample(range(n), rng.randint(1, max(1, n // 5)))
    ff, ft = [a for a, _ in E], [b for _, b in E]
    vals = {f(n, cr, ff, ft) for f in impls.values()}
    assert len(vals) == 1
print('Khớp vét cạn trên', cnt, 'đồ thị nhỏ; 4 cách khớp nhau trên 200 đồ thị vừa.')

Khớp vét cạn trên 300 đồ thị nhỏ; 4 cách khớp nhau trên 200 đồ thị vừa.


### 12.3. Ca lớn `n = 10^5`, `m = 2·10^5`

In [12]:
import time
n = 10 ** 5
cases = {
    'chuỗi dài': (n, [n - 1], list(range(n - 1)), list(range(1, n))),
    'chu trình lớn': (n, [0], list(range(n)), [(i + 1) % n for i in range(n)]),
    'ngẫu nhiên': None,
}
E = list({(rng.randrange(n), rng.randrange(n)) for _ in range(2 * n)})
E = [(a, b) for a, b in E if a != b]
cases['ngẫu nhiên'] = (n, rng.sample(range(n), 50), [a for a, _ in E], [b for _, b in E])
for name, (nn, cr, ff, ft) in cases.items():
    t = time.time(); a = Solution().minRunesToAdd(nn, cr, ff, ft); t1 = time.time() - t
    t = time.time(); b = min_runes_doocs(nn, cr, ff, ft); t2 = time.time() - t
    assert a == b
    print(f"{name:14s}: {a:6d} | Tarjan {t1:.2f}s | lan truyền + DFS {t2:.2f}s")

chuỗi dài     :      1 | Tarjan 0.14s | lan truyền + DFS 0.10s
chu trình lớn :      0 | Tarjan 0.13s | lan truyền + DFS 0.03s


ngẫu nhiên    :  13484 | Tarjan 0.32s | lan truyền + DFS 0.17s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm các **đỉnh** có bậc vào 0 (bỏ qua chu trình)

In [13]:
def wrong_vertex_indeg(n, crystals, ff, ft):
    indeg = [0] * n
    for b in ft:
        indeg[b] += 1
    cs = set(crystals)
    return sum(1 for v in range(n) if indeg[v] == 0 and v not in cs)

n, cr, ff, ft, _ = examples[1]
print('sai :', wrong_vertex_indeg(n, cr, ff, ft), '| đúng:', Solution().minRunesToAdd(n, cr, ff, ft))

sai : 0 | đúng: 1


**Vì sao sai:** chu trình `0 → 1 → 2 → 0` không có đỉnh nào bậc vào 0 nhưng cả chu trình không nhận được năng lượng từ ngoài.

### Lỗi 2 — Đếm số **đỉnh** không được thắp sáng

In [14]:
def wrong_count_dark(n, crystals, ff, ft):
    return sum(1 for v in lit_set(n, crystals, list(zip(ff, ft))) if not v)

print('sai :', wrong_count_dark(n, cr, ff, ft), '| đúng:', Solution().minRunesToAdd(n, cr, ff, ft))

sai : 3 | đúng: 1


**Vì sao sai:** một cạnh mới vào một SCC (hoặc vào một nguồn) thắp sáng **nhiều** đỉnh cùng lúc.

### Lỗi 3 — Đếm mọi SCC nguồn (kể cả SCC đã có tinh thể)

In [15]:
def wrong_all_sources(n, crystals, ff, ft):
    g = [[] for _ in range(n)]
    for a, b in zip(ff, ft):
        g[a].append(b)
    c, comp = tarjan_scc(n, g)
    indeg = [0] * c
    for a, b in zip(ff, ft):
        if comp[a] != comp[b]:
            indeg[comp[b]] += 1
    return sum(1 for k in range(c) if indeg[k] == 0)

print('sai :', wrong_all_sources(n, cr, ff, ft), '| đúng:', Solution().minRunesToAdd(n, cr, ff, ft))

sai : 3 | đúng: 1


**Vì sao sai:** SCC nguồn chứa tinh thể tự phát sáng, không cần cạnh mới.

### Lỗi 4 — Tarjan đệ quy trên chuỗi dài

In [16]:
def tarjan_recursive_depth(n, ff, ft):
    g = [[] for _ in range(n)]
    for a, b in zip(ff, ft):
        g[a].append(b)
    seen = [False] * n
    def dfs(v):
        seen[v] = True
        for w in g[v]:
            if not seen[w]:
                dfs(w)
    dfs(0)

try:
    tarjan_recursive_depth(*cases['chuỗi dài'][0:1], *cases['chuỗi dài'][2:4])
except RecursionError as e:
    print('RecursionError:', e)

RecursionError: maximum recursion depth exceeded


**Vì sao sai:** chuỗi `10^5` đỉnh vượt giới hạn đệ quy mặc định của Python. Dùng ngăn xếp tự quản.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mọi đỉnh phải đi tới được từ một tập nguồn, thêm ít cạnh nhất" ⇒ **SCC + đồ thị cô đặc + đếm nguồn**.
- Bài kinh điển cùng họ: "thêm ít cạnh nhất để đồ thị liên thông mạnh" ⇒ `max(#nguồn, #đích)` (trừ khi chỉ có 1 SCC).
- Tinh thể = các đỉnh "đã có nguồn" ⇒ loại SCC chứa chúng khỏi tập nguồn cần xử lý.
- Python + `n ≥ 10^4` ⇒ DFS/Tarjan không đệ quy.

## 15. Biến thể của bài

1. Thêm ít cạnh nhất để **cả đồ thị liên thông mạnh** ⇒ `max(#SCC nguồn, #SCC đích)` (nếu > 1 SCC).
2. Chỉ cho phép thêm cạnh **từ tinh thể** ⇒ vẫn `K` (cách đạt được đã dùng cạnh từ tinh thể).
3. Mỗi cạnh mới có chi phí phụ thuộc đích ⇒ với mỗi SCC nguồn chọn đỉnh rẻ nhất trong nó.
4. Bài liên quan: LC 1557 (Minimum Number of Vertices to Reach All Nodes — DAG), LC 2858.

In [17]:
# Biến thể 1: số cạnh ít nhất để cả đồ thị liên thông mạnh
def min_edges_strongly_connected(n, ff, ft):
    g = [[] for _ in range(n)]
    for a, b in zip(ff, ft):
        g[a].append(b)
    c, comp = tarjan_scc(n, g)
    if c == 1:
        return 0
    has_in, has_out = [False] * c, [False] * c
    for a, b in zip(ff, ft):
        if comp[a] != comp[b]:
            has_out[comp[a]] = True; has_in[comp[b]] = True
    return max(has_in.count(False), has_out.count(False))

def brute_sc(n, ff, ft):
    E = list(zip(ff, ft)); have = set(E)
    cand = [(a, b) for a in range(n) for b in range(n) if a != b and (a, b) not in have]
    for k in range(n + 1):
        for add in itertools.combinations(cand, k):
            if all(all(lit_set(n, [s], E + list(add))) for s in range(n)):
                return k

rng = random.Random(15)
for _ in range(80):
    n = rng.randint(2, 4)
    pairs = [(a, b) for a in range(n) for b in range(n) if a != b]
    E = rng.sample(pairs, rng.randint(1, len(pairs)))
    assert min_edges_strongly_connected(n, [a for a, _ in E], [b for _, b in E]) == brute_sc(n, [a for a, _ in E], [b for _, b in E])
print('Biến thể "liên thông mạnh" khớp vét cạn.')

Biến thể "liên thông mạnh" khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Thử thêm cạnh | mũ | Oracle |
| 2 | Lan truyền + thứ tự DFS | `O(n + m)` | Lời giải doocs |
| 3 | Tarjan + đếm nguồn | `O(n + m)` | **Nộp bài** |
| 4 | Kosaraju | `O(n + m)` | Thay thế Tarjan |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `n = 3`, tinh thể `[0]`, cạnh `0→1`.

**Bài 2 (tính tay):** `n = 5`, tinh thể `[4]`, cạnh `0→1, 1→0, 2→3`.

**Bài 3 (điền TODO):** trả về **danh sách cạnh** cần thêm (một lời giải tối ưu).

**Bài 4 (tự viết lại):** biến thể 3 — thêm cạnh vào đỉnh `v` tốn `w[v]`; tối thiểu tổng chi phí.

**Bài 5 (mini challenge):** LC 1557 — trên DAG, tìm tập đỉnh nhỏ nhất đi tới được mọi đỉnh.

In [18]:
def edges_to_add_todo(n, crystals, ff, ft):
    # TODO
    raise NotImplementedError

def min_cost_runes_todo(n, crystals, ff, ft, w):
    # TODO
    raise NotImplementedError

def smallest_set_of_vertices_todo(n, edges):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Đỉnh 2 cô lập. (2) Một SCC nguồn không tinh thể. (3) `1`.

**Bài 2.** (1) SCC `{0,1}`, `{2}`, `{3}`, `{4}`. (2) Nguồn: `{0,1}`, `{2}`, `{4}`. (3) `{4}` có tinh thể ⇒ `2`.

**Bài 3.** (1) Lấy một tinh thể `c`. (2) Với mỗi SCC nguồn cần xử lý, chọn một đỉnh `v`. (3) Thêm `(c, v)`; kiểm tra bằng BFS.

**Bài 4.** (1) Mỗi SCC nguồn cần đúng một cạnh vào. (2) Chọn đỉnh có `w` nhỏ nhất trong SCC. (3) Cộng lại.

**Bài 5.** (1) DAG ⇒ SCC là từng đỉnh. (2) Đỉnh bậc vào 0. (3) Không cần tinh thể.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
assert Solution().minRunesToAdd(3, [0], [0], [1]) == 1
assert Solution().minRunesToAdd(5, [4], [0, 1, 2], [1, 0, 3]) == 2 == min_runes_brute(5, [4], [0, 1, 2], [1, 0, 3])

def edges_to_add(n, crystals, ff, ft):
    g = [[] for _ in range(n)]
    for a, b in zip(ff, ft):
        g[a].append(b)
    c, comp = tarjan_scc(n, g)
    need = [True] * c
    for a, b in zip(ff, ft):
        if comp[a] != comp[b]:
            need[comp[b]] = False
    for x in crystals:
        need[comp[x]] = False
    rep = {}
    for v in range(n):
        if need[comp[v]] and comp[v] not in rep:
            rep[comp[v]] = v
    return [(crystals[0], v) for v in rep.values()]

rng = random.Random(3)
for _ in range(200):
    n = rng.randint(2, 30)
    E = list({(rng.randrange(n), rng.randrange(n)) for _ in range(rng.randint(1, 2 * n))})
    E = [(a, b) for a, b in E if a != b] or [(0, 1)]
    cr = rng.sample(range(n), rng.randint(1, min(3, n)))
    add = edges_to_add(n, cr, [a for a, _ in E], [b for _, b in E])
    assert len(add) == Solution().minRunesToAdd(n, cr, [a for a, _ in E], [b for _, b in E])
    assert all(lit_set(n, cr, E + add))
print('Bài 1: 1 | Bài 2: 2 | Bài 3: danh sách cạnh hợp lệ; ví dụ 2 ->', edges_to_add(*examples[1][:4]))

Bài 1: 1 | Bài 2: 2 | Bài 3: danh sách cạnh hợp lệ; ví dụ 2 -> [(3, 0)]


In [20]:
# Bài 4, 5
def min_cost_runes(n, crystals, ff, ft, w):
    g = [[] for _ in range(n)]
    for a, b in zip(ff, ft):
        g[a].append(b)
    c, comp = tarjan_scc(n, g)
    need = [True] * c
    for a, b in zip(ff, ft):
        if comp[a] != comp[b]:
            need[comp[b]] = False
    for x in crystals:
        need[comp[x]] = False
    best = {}
    for v in range(n):
        k = comp[v]
        if need[k]:
            best[k] = min(best.get(k, float('inf')), w[v])
    return sum(best.values())

def smallest_set_of_vertices(n, edges):
    indeg = [0] * n
    for _, b in edges:
        indeg[b] += 1
    return [v for v in range(n) if indeg[v] == 0]

assert min_cost_runes(7, [3, 5], [0, 1, 2, 3, 5], [1, 2, 0, 4, 6], [5, 2, 9, 0, 0, 0, 0]) == 2
assert smallest_set_of_vertices(6, [[0, 1], [0, 2], [2, 5], [3, 4], [4, 2]]) == [0, 3]         # LC 1557 ví dụ
assert smallest_set_of_vertices(5, [[0, 1], [2, 1], [3, 1], [1, 4], [2, 4]]) == [0, 2, 3]
print('Bài 4, 5 đúng.')

Bài 4, 5 đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
comp = SCC (Tarjan/Kosaraju không đệ quy)
has_in[k] = có cạnh (a, b) với comp[a] != comp[b] = k   hoặc   k chứa tinh thể
return số k với has_in[k] == False
```

**Checklist:**

- [ ] Làm việc trên **SCC**, không phải từng đỉnh?
- [ ] Chỉ đếm SCC **bậc vào 0** trong đồ thị cô đặc?
- [ ] Loại SCC chứa tinh thể?
- [ ] DFS không đệ quy?
- [ ] Đối chiếu chữ ký nếu có Premium?